In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 读取数据
df = pd.read_csv('complete_synced_data.csv')

print("=" * 50)
print("数据基本信息")
print("=" * 50)
print(f"数据形状: {df.shape}")
print(f"\n数据类型:")
print(df.dtypes)

print(f"\n前5行数据:")
print(df.head())

print(f"\n缺失值情况:")
print(df.isnull().sum())

print(f"\n数值型特征描述性统计:")
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(df[numeric_cols].describe())

In [ ]:
# 第二步：目标变量分析和特征相关性分析

print("=" * 60)
print("目标变量（能见度指标）分析")
print("=" * 60)

# 分析4个能见度指标
visibility_cols = ['visibility_vis_1a', 'visibility_vis_10a', 'visibility_mor_raw', 'visibility_vis_raw']

print("能见度指标描述性统计:")
print(df[visibility_cols].describe())

print("\n能见度指标之间的相关性:")
vis_corr = df[visibility_cols].corr()
print(vis_corr)

# 绘制能见度指标分布图
plt.figure(figsize=(15, 10))
for i, col in enumerate(visibility_cols, 1):
    plt.subplot(2, 2, i)
    plt.hist(df[col], bins=50, alpha=0.7, edgecolor='black')
    plt.title(f'{col} 分布直方图')
    plt.xlabel('能见度值')
    plt.ylabel('频次')
    
    # 添加统计信息
    mean_val = df[col].mean()
    std_val = df[col].std()
    plt.axvline(mean_val, color='red', linestyle='--', label=f'均值: {mean_val:.1f}')
    plt.legend()

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("图像特征与主要能见度指标相关性分析")
print("=" * 60)

# 选择图像特征
image_features = ['laplacian_var', 'sobel_mean', 'sobel_std', 'contrast_std', 
                 'high_freq_ratio', 'edge_density', 'entropy', 'local_var_mean', 'brenner']

# 选择visibility_mor_raw作为主要目标变量（这是标准的气象光学视程）
target = 'visibility_mor_raw'

print(f"图像特征与{target}的相关系数:")
correlations = {}
for feature in image_features:
    corr_coef = df[feature].corr(df[target])
    correlations[feature] = corr_coef
    print(f"{feature:<20}: {corr_coef:7.4f}")

print("\n相关系数排序（绝对值）:")
sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
for feature, corr in sorted_corr:
    print(f"{feature:<20}: {corr:7.4f}")

print("\n" + "=" * 60)
print("气象要素与能见度相关性分析")
print("=" * 60)

# 选择气象特征
weather_features = ['weather_temperature_c', 'weather_humidity_pct', 'weather_dewpoint_c',
                   'weather_pressure_hpa', 'weather_qnh_hpa',
                   'wind_wind_speed_2m', 'wind_wind_speed_10m', 'wind_gust_speed']

print(f"气象要素与{target}的相关系数:")
weather_correlations = {}
for feature in weather_features:
    corr_coef = df[feature].corr(df[target])
    weather_correlations[feature] = corr_coef
    print(f"{feature:<25}: {corr_coef:7.4f}")

print("\n气象要素相关系数排序（绝对值）:")
sorted_weather_corr = sorted(weather_correlations.items(), key=lambda x: abs(x[1]), reverse=True)
for feature, corr in sorted_weather_corr:
    print(f"{feature:<25}: {corr:7.4f}")

# 检查极端值
print("\n" + "=" * 60)
print("异常值检查")
print("=" * 60)

print(f"{target}的极值情况:")
print(f"最小值: {df[target].min():.2f}")
print(f"最大值: {df[target].max():.2f}")
print(f"均值: {df[target].mean():.2f}")
print(f"中位数: {df[target].median():.2f}")

# 检查异常低的能见度值（可能的大雾情况）
low_vis_threshold = df[target].quantile(0.1)  # 10分位数
high_vis_threshold = df[target].quantile(0.9)  # 90分位数

print(f"\n能见度分布情况:")
print(f"10分位数 (低能见度): {low_vis_threshold:.2f}")
print(f"90分位数 (高能见度): {high_vis_threshold:.2f}")

low_vis_count = len(df[df[target] <= low_vis_threshold])
high_vis_count = len(df[df[target] >= high_vis_threshold])

print(f"低能见度样本数 (<= {low_vis_threshold:.2f}): {low_vis_count}")
print(f"高能见度样本数 (>= {high_vis_threshold:.2f}): {high_vis_count}")

In [ ]:
# 第三步：特征选择和数据预处理

print("=" * 60)
print("特征选择和数据预处理")
print("=" * 60)

# 基于相关性分析选择特征
# 选择相关性较强的图像特征（|相关系数| > 0.35）
selected_image_features = ['high_freq_ratio', 'laplacian_var', 'edge_density', 
                          'sobel_mean', 'local_var_mean', 'contrast_std', 'brenner', 'sobel_std']

# 选择重要的气象特征
selected_weather_features = ['weather_humidity_pct', 'weather_temperature_c', 
                           'weather_pressure_hpa', 'wind_wind_speed_10m', 'wind_wind_speed_2m']

# 目标变量
target = 'visibility_mor_raw'

# 组合所有选择的特征
all_features = selected_image_features + selected_weather_features

print("选择的特征:")
print("图像特征:", selected_image_features)
print("气象特征:", selected_weather_features)
print("目标变量:", target)
print("总特征数:", len(all_features))

# 创建建模数据集
X = df[all_features].copy()
y = df[target].copy()

print(f"\n建模数据集形状:")
print(f"特征矩阵 X: {X.shape}")
print(f"目标变量 y: {y.shape}")

# 检查多重共线性
print("\n" + "=" * 60)
print("特征间相关性检查（检查多重共线性）")
print("=" * 60)

feature_corr = X.corr()
print("特征相关系数矩阵:")
print(feature_corr.round(3))

# 找出高相关的特征对
high_corr_pairs = []
for i in range(len(feature_corr.columns)):
    for j in range(i+1, len(feature_corr.columns)):
        corr_val = abs(feature_corr.iloc[i, j])
        if corr_val > 0.8:  # 相关系数绝对值大于0.8认为存在多重共线性
            high_corr_pairs.append((feature_corr.columns[i], feature_corr.columns[j], corr_val))

print(f"\n高相关特征对 (|相关系数| > 0.8):")
if high_corr_pairs:
    for feat1, feat2, corr in high_corr_pairs:
        print(f"{feat1} - {feat2}: {corr:.3f}")
else:
    print("未发现严重的多重共线性问题")

# 数据标准化前的统计
print("\n" + "=" * 60)
print("数据标准化前的统计")
print("=" * 60)
print("特征的量纲差异:")
for feature in all_features:
    mean_val = X[feature].mean()
    std_val = X[feature].std()
    min_val = X[feature].min()
    max_val = X[feature].max()
    print(f"{feature:<25}: 均值={mean_val:10.2f}, 标准差={std_val:10.2f}, 范围=[{min_val:8.2f}, {max_val:10.2f}]")

# 数据标准化
print("\n进行数据标准化...")
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=all_features, index=X.index)

# 对目标变量进行对数变换（因为能见度数据通常呈偏态分布）
print("\n检查目标变量分布，考虑是否需要变换...")
y_stats = {
    '原始': y,
    'log变换': np.log(y + 1),  # 加1避免log(0)
    'sqrt变换': np.sqrt(y)
}

print("目标变量变换后的分布统计:")
for name, transformed_y in y_stats.items():
    skewness = stats.skew(transformed_y)
    kurtosis = stats.kurtosis(transformed_y)
    print(f"{name:<10}: 偏度={skewness:6.3f}, 峰度={kurtosis:6.3f}")

# 选择偏度最接近0的变换
best_transform = min(y_stats.items(), key=lambda x: abs(stats.skew(x[1])))
print(f"\n选择变换方式: {best_transform[0]} (偏度最接近0)")

if best_transform[0] == 'log变换':
    y_transformed = np.log(y + 1)
    transform_type = 'log'
elif best_transform[0] == 'sqrt变换':
    y_transformed = np.sqrt(y)
    transform_type = 'sqrt'
else:
    y_transformed = y
    transform_type = 'none'

print(f"使用的目标变量变换: {transform_type}")

# 数据分割
print("\n" + "=" * 60)
print("数据分割")
print("=" * 60)

# 分割数据集 (80%训练，20%测试)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_transformed, test_size=0.2, random_state=42, stratify=None
)

print(f"训练集大小: {X_train.shape[0]} 样本")
print(f"测试集大小: {X_test.shape[0]} 样本")
print(f"训练集目标变量范围: [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"测试集目标变量范围: [{y_test.min():.2f}, {y_test.max():.2f}]")

# 保存预处理信息，用于后续建模
preprocessing_info = {
    'selected_features': all_features,
    'image_features': selected_image_features,
    'weather_features': selected_weather_features,
    'target': target,
    'transform_type': transform_type,
    'scaler_X': scaler_X,
    'train_size': len(X_train),
    'test_size': len(X_test)
}

print(f"\n预处理完成，准备建模...")
print(f"特征标准化: 完成")
print(f"目标变量变换: {transform_type}")
print(f"数据分割: 完成")

In [ ]:
# 第四步：处理多重共线性并建立回归模型

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

print("=" * 70)
print("多重共线性处理和特征选择")
print("=" * 70)

# 方法1：基于VIF (方差膨胀因子) 选择特征
def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data["特征"] = df.columns
    vif_data["VIF"] = [variance_inflation_factor(df.values, i) for i in range(len(df.columns))]
    return vif_data.sort_values('VIF', ascending=False)

print("计算方差膨胀因子(VIF):")
vif_scores = calculate_vif(X_train)
print(vif_scores)
print("\n注：VIF > 10 表示存在严重多重共线性")

# 方法2：基于相关性手动选择代表性特征
print("\n" + "=" * 70)
print("手动选择代表性特征（避免高相关性）")
print("=" * 70)

# 基于相关性分析，从每组高相关特征中选择一个代表
selected_features_manual = [
    'laplacian_var',        # 代表图像清晰度组 (与local_var_mean完全相关，选一个)
    'high_freq_ratio',      # 高频信息比例（相对独立）
    'edge_density',         # 边缘密度（代表边缘特征组）
    'contrast_std',         # 对比度标准差
    'weather_humidity_pct', # 湿度（气象要素中最重要）
    'weather_temperature_c',# 温度（与湿度高相关但物理意义不同）
    'wind_wind_speed_10m'   # 风速（相对独立的气象要素）
]

print("手动选择的特征:")
for i, feat in enumerate(selected_features_manual, 1):
    print(f"{i}. {feat}")

# 检查手动选择特征间的相关性
X_manual = X_train[selected_features_manual]
manual_corr = X_manual.corr()
print(f"\n手动选择特征间的相关性矩阵:")
print(manual_corr.round(3))

# 计算手动选择特征的VIF
vif_manual = calculate_vif(X_manual)
print(f"\n手动选择特征的VIF:")
print(vif_manual)

print("\n" + "=" * 70)
print("建立多种回归模型")
print("=" * 70)

# 使用手动选择的特征建立多种模型
X_train_selected = X_train[selected_features_manual]
X_test_selected = X_test[selected_features_manual]

models = {
    '普通线性回归': LinearRegression(),
    '岭回归(Ridge)': Ridge(alpha=1.0),
    'Lasso回归': Lasso(alpha=0.1),
    '弹性网络': ElasticNet(alpha=0.1, l1_ratio=0.5)
}

# 训练和评估模型
model_results = {}
for name, model in models.items():
    print(f"\n{'='*20} {name} {'='*20}")
    
    # 训练模型
    model.fit(X_train_selected, y_train)
    
    # 预测
    y_train_pred = model.predict(X_train_selected)
    y_test_pred = model.predict(X_test_selected)
    
    # 评估指标
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    print(f"训练集 R²: {train_r2:.4f}")
    print(f"测试集 R²: {test_r2:.4f}")
    print(f"训练集 RMSE: {train_rmse:.4f}")
    print(f"测试集 RMSE: {test_rmse:.4f}")
    print(f"训练集 MAE: {train_mae:.4f}")
    print(f"测试集 MAE: {test_mae:.4f}")
    
    # 保存结果
    model_results[name] = {
        'model': model,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'coefficients': getattr(model, 'coef_', None),
        'intercept': getattr(model, 'intercept_', None)
    }

# 选择最佳模型（基于测试集R²）
best_model_name = max(model_results.keys(), key=lambda k: model_results[k]['test_r2'])
best_model = model_results[best_model_name]

print(f"\n" + "=" * 70)
print("最佳模型和数学公式")
print("=" * 70)

print(f"最佳模型: {best_model_name}")
print(f"测试集 R²: {best_model['test_r2']:.4f}")
print(f"测试集 RMSE: {best_model['test_rmse']:.4f}")

# 建立明确的数学公式
print(f"\n数学模型公式:")
print("=" * 50)

if best_model['coefficients'] is not None:
    print("标准化后的特征回归方程:")
    print("sqrt(能见度) = β₀ + β₁×X₁ + β₂×X₂ + ... + β₇×X₇")
    print()
    print("其中:")
    print(f"β₀ (截距) = {best_model['intercept']:.6f}")
    
    for i, (feat, coef) in enumerate(zip(selected_features_manual, best_model['coefficients']), 1):
        print(f"β{i} ({feat}) = {coef:8.6f}")
    
    print(f"\n完整公式:")
    formula_parts = [f"{best_model['intercept']:.6f}"]
    for feat, coef in zip(selected_features_manual, best_model['coefficients']):
        sign = "+" if coef >= 0 else ""
        formula_parts.append(f"{sign}{coef:.6f}×{feat}")
    
    print("sqrt(visibility_mor_raw) = " + " ".join(formula_parts))
    
    print(f"\n因此，能见度预测公式为:")
    print("visibility_mor_raw = [上述公式结果]²")

# 特征重要性分析
print(f"\n" + "=" * 70)
print("特征重要性分析")
print("=" * 70)

if best_model['coefficients'] is not None:
    # 计算标准化系数的绝对值作为重要性指标
    feature_importance = pd.DataFrame({
        '特征': selected_features_manual,
        '回归系数': best_model['coefficients'],
        '重要性(|系数|)': np.abs(best_model['coefficients'])
    }).sort_values('重要性(|系数|)', ascending=False)
    
    print("特征重要性排序:")
    print(feature_importance)
    
    # 分析系数符号的物理意义
    print(f"\n系数符号的物理意义:")
    for feat, coef in zip(selected_features_manual, best_model['coefficients']):
        direction = "正相关" if coef > 0 else "负相关"
        print(f"{feat:<25}: {coef:8.6f} ({direction})")

In [ ]:
# 第五步：模型验证、残差分析和最终公式

print("=" * 70)
print("模型验证和残差分析")
print("=" * 70)

# 获取最佳模型（Lasso回归）
best_model = model_results['Lasso回归']['model']

# 获取有效特征（系数不为0的特征）
active_features = []
active_coefficients = []
for feat, coef in zip(selected_features_manual, best_model.coef_):
    if abs(coef) > 1e-10:  # 系数不为0
        active_features.append(feat)
        active_coefficients.append(coef)

print("Lasso回归选择的有效特征:")
for feat, coef in zip(active_features, active_coefficients):
    print(f"{feat:<25}: {coef:10.6f}")

# 重新训练只使用有效特征的简化模型
X_train_final = X_train[active_features]
X_test_final = X_test[active_features]

final_model = LinearRegression()
final_model.fit(X_train_final, y_train)

# 预测
y_train_pred_final = final_model.predict(X_train_final)
y_test_pred_final = final_model.predict(X_test_final)

# 最终模型性能
final_train_r2 = r2_score(y_train, y_train_pred_final)
final_test_r2 = r2_score(y_test, y_test_pred_final)
final_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_final))
final_test_mae = mean_absolute_error(y_test, y_test_pred_final)

print(f"\n最终简化模型性能:")
print(f"训练集 R²: {final_train_r2:.4f}")
print(f"测试集 R²: {final_test_r2:.4f}")
print(f"测试集 RMSE: {final_test_rmse:.4f} (标准化sqrt尺度)")
print(f"测试集 MAE: {final_test_mae:.4f} (标准化sqrt尺度)")

print("\n" + "=" * 70)
print("最终数学公式（标准化特征）")
print("=" * 70)

print("标准化特征的回归方程:")
print("sqrt(visibility_mor_raw) = β₀ + Σ(βᵢ × Xᵢ_standardized)")
print()
print("其中：")
print(f"β₀ (截距) = {final_model.intercept_:.6f}")

formula_parts = [f"{final_model.intercept_:.6f}"]
for i, (feat, coef) in enumerate(zip(active_features, final_model.coef_), 1):
    print(f"β{i} ({feat}) = {coef:10.6f}")
    sign = "+" if coef >= 0 else ""
    formula_parts.append(f"{sign}{coef:.6f}×{feat}_std")

print(f"\n完整标准化公式:")
print("sqrt(visibility_mor_raw) = " + " ".join(formula_parts))

print("\n" + "=" * 70)
print("转换为原始特征尺度的公式")
print("=" * 70)

# 获取标准化参数
feature_means = []
feature_stds = []
for feat in active_features:
    idx = list(X.columns).index(feat)
    mean_val = scaler_X.mean_[idx]
    std_val = scaler_X.scale_[idx]
    feature_means.append(mean_val)
    feature_stds.append(std_val)
    print(f"{feat}:")
    print(f"  均值 = {mean_val:.6f}")
    print(f"  标准差 = {std_val:.6f}")

# 计算原始尺度的系数和截距
original_intercept = final_model.intercept_
for coef, mean_val, std_val in zip(final_model.coef_, feature_means, feature_stds):
    original_intercept -= coef * mean_val / std_val

original_coefficients = []
for coef, std_val in zip(final_model.coef_, feature_stds):
    original_coefficients.append(coef / std_val)

print(f"\n原始尺度的回归方程:")
print("sqrt(visibility_mor_raw) = α₀ + Σ(αᵢ × Xᵢ_original)")
print()
print("其中：")
print(f"α₀ (截距) = {original_intercept:.6f}")

original_formula_parts = [f"{original_intercept:.6f}"]
for i, (feat, coef) in enumerate(zip(active_features, original_coefficients), 1):
    print(f"α{i} ({feat}) = {coef:.10f}")
    sign = "+" if coef >= 0 else ""
    original_formula_parts.append(f"{sign}{coef:.6e}×{feat}")

print(f"\n完整原始尺度公式:")
print("sqrt(visibility_mor_raw) = " + " ".join(original_formula_parts))

print(f"\n最终能见度预测公式:")
print("visibility_mor_raw = [上述sqrt结果]²")

print("\n" + "=" * 70)
print("残差分析")
print("=" * 70)

# 残差分析
residuals_train = y_train - y_train_pred_final
residuals_test = y_test - y_test_pred_final

print("残差统计:")
print(f"训练集残差 - 均值: {residuals_train.mean():.6f}, 标准差: {residuals_train.std():.6f}")
print(f"测试集残差 - 均值: {residuals_test.mean():.6f}, 标准差: {residuals_test.std():.6f}")

# 绘制残差图
plt.figure(figsize=(15, 10))

# 预测值vs残差图
plt.subplot(2, 3, 1)
plt.scatter(y_test_pred_final, residuals_test, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('预测值')
plt.ylabel('残差')
plt.title('预测值 vs 残差')

# 残差直方图
plt.subplot(2, 3, 2)
plt.hist(residuals_test, bins=30, alpha=0.7, edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--')
plt.xlabel('残差')
plt.ylabel('频次')
plt.title('残差分布直方图')

# Q-Q图（正态性检验）
from scipy import stats
plt.subplot(2, 3, 3)
stats.probplot(residuals_test, dist="norm", plot=plt)
plt.title('残差Q-Q图（正态性检验）')

# 真实值vs预测值图
plt.subplot(2, 3, 4)
plt.scatter(y_test, y_test_pred_final, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'red', linestyle='--')
plt.xlabel('真实值 (sqrt尺度)')
plt.ylabel('预测值 (sqrt尺度)')
plt.title('真实值 vs 预测值')

# 转换回原始尺度进行比较
y_test_original = y_test**2  # sqrt逆变换
y_test_pred_original = y_test_pred_final**2

plt.subplot(2, 3, 5)
plt.scatter(y_test_original, y_test_pred_original, alpha=0.6)
plt.plot([y_test_original.min(), y_test_original.max()], 
         [y_test_original.min(), y_test_original.max()], 'red', linestyle='--')
plt.xlabel('真实能见度 (米)')
plt.ylabel('预测能见度 (米)')
plt.title('原始尺度: 真实值 vs 预测值')

# 计算原始尺度的精度
original_r2 = r2_score(y_test_original, y_test_pred_original)
original_rmse = np.sqrt(mean_squared_error(y_test_original, y_test_pred_original))
original_mae = mean_absolute_error(y_test_original, y_test_pred_original)

plt.subplot(2, 3, 6)
plt.text(0.1, 0.8, f'原始尺度模型精度:', fontsize=14, fontweight='bold')
plt.text(0.1, 0.7, f'R² = {original_r2:.4f}', fontsize=12)
plt.text(0.1, 0.6, f'RMSE = {original_rmse:.1f} 米', fontsize=12)
plt.text(0.1, 0.5, f'MAE = {original_mae:.1f} 米', fontsize=12)
plt.text(0.1, 0.3, f'平均相对误差:', fontsize=12)
mape = np.mean(np.abs((y_test_original - y_test_pred_original) / y_test_original)) * 100
plt.text(0.1, 0.2, f'MAPE = {mape:.1f}%', fontsize=12)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"\n原始尺度（米）的模型精度:")
print(f"R² = {original_r2:.4f}")
print(f"RMSE = {original_rmse:.1f} 米")
print(f"MAE = {original_mae:.1f} 米")
print(f"MAPE = {mape:.1f}%")

print("\n" + "=" * 70)
print("模型总结")
print("=" * 70)

print("1. 最终选择的特征数量:", len(active_features))
print("2. 模型类型: 多元线性回归 (经Lasso特征选择)")
print("3. 目标变量变换: 平方根变换")
print("4. 特征标准化: Z-score标准化")
print(f"5. 模型精度: R² = {original_r2:.4f}, RMSE = {original_rmse:.1f}米")
print(f"6. 平均绝对百分比误差: {mape:.1f}%")

In [ ]:
# 改进

In [ ]:
# 高级建模方案：基于物理原理的两步法 + 非线性模型

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

print("=" * 80)
print("高级建模方案")  
print("=" * 80)

# 方案一：基于物理原理的两步建模法
print("\n【方案一：基于物理原理的两步建模法】")
print("理论基础：MOR = 3.912/σ，其中σ为衰减系数")
print("建模思路：图像特征+气象要素 → σ → MOR")

# 从已知的MOR计算衰减系数σ
sigma = 3.912 / df['visibility_mor_raw']
print(f"\n衰减系数σ统计:")
print(f"均值: {sigma.mean():.6f}")
print(f"标准差: {sigma.std():.6f}")
print(f"范围: [{sigma.min():.6f}, {sigma.max():.6f}]")

# 建立图像特征与衰减系数的关系
image_features = ['laplacian_var', 'high_freq_ratio', 'edge_density', 'sobel_mean', 'local_var_mean']
weather_features = ['weather_humidity_pct', 'weather_temperature_c', 'wind_wind_speed_10m']

# 分析σ与各特征的关系
print(f"\n衰减系数σ与各特征的相关性:")
for feat in image_features + weather_features:
    corr = sigma.corr(df[feat])
    print(f"{feat:<25}: {corr:7.4f}")

# 方案二：非线性模型 - 多项式回归
print(f"\n【方案二：非线性多项式回归模型】")

# 选择关键特征
key_features = ['laplacian_var', 'high_freq_ratio', 'weather_humidity_pct', 
                'weather_temperature_c', 'wind_wind_speed_10m']

X_key = df[key_features]
y = df['visibility_mor_raw']

# 创建多项式特征（包括交互项）
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X_key)
feature_names_poly = poly.get_feature_names_out(key_features)

print(f"原始特征数: {len(key_features)}")
print(f"多项式特征数: {len(feature_names_poly)}")

# 拆分训练测试集
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)

# 训练多项式回归模型
poly_model = LinearRegression()
poly_model.fit(X_train, y_train)

y_pred_poly = poly_model.predict(X_test)
r2_poly = r2_score(y_test, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))

print(f"多项式回归性能:")
print(f"R² = {r2_poly:.4f}")
print(f"RMSE = {rmse_poly:.1f} 米")

# 提取重要的多项式项
poly_coeffs = pd.DataFrame({
    '特征': feature_names_poly,
    '系数': poly_model.coef_
})
poly_coeffs['重要性'] = np.abs(poly_coeffs['系数'])
top_poly_features = poly_coeffs.nlargest(10, '重要性')

print(f"\n最重要的10个多项式项:")
for idx, row in top_poly_features.iterrows():
    print(f"{row['特征']:<40}: {row['系数']:10.6f}")

# 方案三：分段回归模型
print(f"\n【方案三：基于气象条件的分段回归模型】")

# 根据湿度分段（雾的主要影响因子）
humidity_threshold_low = df['weather_humidity_pct'].quantile(0.33)   # 33分位数
humidity_threshold_high = df['weather_humidity_pct'].quantile(0.67)  # 67分位数

print(f"湿度分段阈值:")
print(f"低湿度: < {humidity_threshold_low:.1f}%")
print(f"中湿度: {humidity_threshold_low:.1f}% - {humidity_threshold_high:.1f}%") 
print(f"高湿度: > {humidity_threshold_high:.1f}%")

# 创建分段数据
low_humidity_mask = df['weather_humidity_pct'] < humidity_threshold_low
mid_humidity_mask = (df['weather_humidity_pct'] >= humidity_threshold_low) & (df['weather_humidity_pct'] <= humidity_threshold_high)
high_humidity_mask = df['weather_humidity_pct'] > humidity_threshold_high

segments = {
    '低湿度': df[low_humidity_mask],
    '中湿度': df[mid_humidity_mask], 
    '高湿度': df[high_humidity_mask]
}

segment_models = {}
for segment_name, segment_data in segments.items():
    if len(segment_data) > 50:  # 确保有足够样本
        X_seg = segment_data[key_features]
        y_seg = segment_data['visibility_mor_raw']
        
        # 标准化
        from sklearn.preprocessing import StandardScaler
        scaler_seg = StandardScaler()
        X_seg_scaled = scaler_seg.fit_transform(X_seg)
        
        # 训练模型
        model_seg = LinearRegression()
        model_seg.fit(X_seg_scaled, y_seg)
        
        # 评估
        y_pred_seg = model_seg.predict(X_seg_scaled)
        r2_seg = r2_score(y_seg, y_pred_seg)
        
        segment_models[segment_name] = {
            'model': model_seg,
            'scaler': scaler_seg,
            'r2': r2_seg,
            'n_samples': len(segment_data),
            'coefficients': dict(zip(key_features, model_seg.coef_)),
            'intercept': model_seg.intercept_
        }
        
        print(f"\n{segment_name}段模型 (n={len(segment_data)}):")
        print(f"  R² = {r2_seg:.4f}")
        print(f"  截距 = {model_seg.intercept_:.2f}")
        for feat, coef in zip(key_features, model_seg.coef_):
            print(f"  {feat:<25}: {coef:8.4f}")

# 方案四：基于衰减系数的物理建模
print(f"\n【方案四：基于衰减系数的物理建模】")

# 第一步：预测衰减系数σ
sigma_model = LinearRegression()
X_sigma = df[key_features]
scaler_sigma = StandardScaler()
X_sigma_scaled = scaler_sigma.fit_transform(X_sigma)

sigma_model.fit(X_sigma_scaled, sigma)
sigma_pred = sigma_model.predict(X_sigma_scaled)

# 第二步：通过物理公式计算MOR
mor_physical = 3.912 / sigma_pred

# 评估物理模型
r2_physical = r2_score(df['visibility_mor_raw'], mor_physical)
rmse_physical = np.sqrt(mean_squared_error(df['visibility_mor_raw'], mor_physical))

print(f"基于物理原理的两步法:")
print(f"第一步 - σ预测模型:")
print(f"  R²(σ预测) = {r2_score(sigma, sigma_pred):.4f}")
print(f"第二步 - MOR物理计算:")
print(f"  R²(MOR) = {r2_physical:.4f}")
print(f"  RMSE = {rmse_physical:.1f} 米")

print(f"\nσ预测模型系数:")
for feat, coef in zip(key_features, sigma_model.coef_):
    print(f"  {feat:<25}: {coef:10.6f}")

# 方案五：集成模型
print(f"\n【方案五：多模型集成】")

# 首先建立简单的线性基准模型
from sklearn.model_selection import train_test_split
X_base = df[key_features]
y_base = df['visibility_mor_raw']

# 数据分割
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

# 标准化
from sklearn.preprocessing import StandardScaler
scaler_base = StandardScaler()
X_train_scaled = scaler_base.fit_transform(X_train_base)
X_test_scaled = scaler_base.transform(X_test_base)

# 训练简单线性模型
linear_base = LinearRegression()
linear_base.fit(X_train_scaled, y_train_base)
linear_pred = linear_base.predict(X_test_scaled)

# 收集所有模型的预测结果进行集成
predictions = {
    '线性模型': linear_pred,
    '多项式模型': y_pred_poly[:len(y_test_base)] if len(y_pred_poly) >= len(y_test_base) else np.resize(y_pred_poly, len(y_test_base)),
    '物理模型': 3.912 / sigma_model.predict(scaler_sigma.transform(X_test_base))
}

# 简单平均集成
ensemble_pred = np.mean([pred for pred in predictions.values()], axis=0)
r2_ensemble = r2_score(y_test_base, ensemble_pred)
rmse_ensemble = np.sqrt(mean_squared_error(y_test_base, ensemble_pred))

print(f"集成模型性能:")
print(f"R² = {r2_ensemble:.4f}")
print(f"RMSE = {rmse_ensemble:.1f} 米")

print(f"\n各子模型性能对比:")
for name, pred in predictions.items():
    r2_sub = r2_score(y_test_base, pred)
    rmse_sub = np.sqrt(mean_squared_error(y_test_base, pred))
    print(f"{name:<15}: R²={r2_sub:.4f}, RMSE={rmse_sub:.1f}米")

# 绘制模型比较图
plt.figure(figsize=(15, 10))

# 各模型预测效果对比
plt.subplot(2, 3, 1)
plt.scatter(y_test_base, predictions['线性模型'], alpha=0.6, label='线性模型')
plt.plot([y_test_base.min(), y_test_base.max()], [y_test_base.min(), y_test_base.max()], 'r--')
plt.xlabel('真实能见度')
plt.ylabel('预测能见度')
plt.title('线性模型')
plt.legend()

plt.subplot(2, 3, 2)
plt.scatter(y_test_base, predictions['多项式模型'], alpha=0.6, label='多项式模型', color='orange')
plt.plot([y_test_base.min(), y_test_base.max()], [y_test_base.min(), y_test_base.max()], 'r--')
plt.xlabel('真实能见度')
plt.ylabel('预测能见度')
plt.title('多项式模型')
plt.legend()

plt.subplot(2, 3, 3)
plt.scatter(y_test_base, predictions['物理模型'], alpha=0.6, label='物理模型', color='green')
plt.plot([y_test_base.min(), y_test_base.max()], [y_test_base.min(), y_test_base.max()], 'r--')
plt.xlabel('真实能见度')
plt.ylabel('预测能见度')
plt.title('物理模型')
plt.legend()

plt.subplot(2, 3, 4)
plt.scatter(y_test_base, ensemble_pred, alpha=0.6, label='集成模型', color='purple')
plt.plot([y_test_base.min(), y_test_base.max()], [y_test_base.min(), y_test_base.max()], 'r--')
plt.xlabel('真实能见度')
plt.ylabel('预测能见度')
plt.title('集成模型')
plt.legend()

# 模型性能柱状图
plt.subplot(2, 3, 5)
model_names = ['线性', '多项式', '物理', '集成']
r2_scores = [r2_score(y_test_base, pred) for pred in list(predictions.values()) + [ensemble_pred]]
plt.bar(model_names, r2_scores, color=['blue', 'orange', 'green', 'purple'])
plt.ylabel('R²')
plt.title('模型R²对比')
plt.ylim(0.8, 1.0)

# 误差分布
plt.subplot(2, 3, 6)
errors = ensemble_pred - y_test_base
plt.hist(errors, bins=30, alpha=0.7, edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--')
plt.xlabel('预测误差 (米)')
plt.ylabel('频次')
plt.title('集成模型误差分布')

plt.tight_layout()
plt.show()

print(f"\n" + "="*80)
print("高级建模方案总结")
print("="*80)
print("1. 物理原理两步法: 基于衰减系数σ的理论建模")
print("2. 多项式回归: 考虑特征间非线性关系和交互效应") 
print("3. 分段回归: 根据湿度条件建立不同的子模型")
print("4. 物理建模: σ预测 + MOR物理计算")
print("5. 集成模型: 多模型融合提升预测精度")

In [ ]:
# 最终优化模型：基于多项式回归的科学建模方案

print("=" * 80)
print("最终优化建模方案：多项式回归 + 物理解释")
print("=" * 80)

# 基于之前结果，多项式回归表现最佳 (R² = 0.9524)
# 现在我们要给出完整的科学建模方案

# 重新建立多项式模型并提供完整解释
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 核心特征
key_features = ['laplacian_var', 'high_freq_ratio', 'weather_humidity_pct', 
                'weather_temperature_c', 'wind_wind_speed_10m']

# 准备数据
X = df[key_features].copy()
y = df['visibility_mor_raw'].copy()

# 数据分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"数据准备完成:")
print(f"训练集: {X_train.shape[0]} 样本")
print(f"测试集: {X_test.shape[0]} 样本")

# 创建多项式特征
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"多项式特征扩展:")
print(f"原始特征: {X_train_scaled.shape[1]}")
print(f"多项式特征: {X_train_poly.shape[1]}")

# 使用Ridge回归防止过拟合
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_poly, y_train)

# 预测
y_train_pred = ridge_model.predict(X_train_poly)
y_test_pred = ridge_model.predict(X_test_poly)

# 评估
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f"\n最终多项式回归模型性能:")
print(f"训练集 R² = {train_r2:.4f}")
print(f"测试集 R² = {test_r2:.4f}")
print(f"测试集 RMSE = {test_rmse:.1f} 米")
print(f"测试集 MAE = {test_mae:.1f} 米")

# 计算MAPE
mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100
print(f"测试集 MAPE = {mape:.1f}%")

print("\n" + "=" * 80)
print("模型特征重要性分析")
print("=" * 80)

# 获取特征名称和系数
feature_names = poly.get_feature_names_out(key_features)
coefficients = ridge_model.coef_

# 创建重要性数据框
importance_df = pd.DataFrame({
    '特征': feature_names,
    '系数': coefficients,
    '重要性': np.abs(coefficients)
}).sort_values('重要性', ascending=False)

print("最重要的15个特征项:")
top_features = importance_df.head(15)
for idx, row in top_features.iterrows():
    print(f"{row['特征']:<40}: {row['系数']:10.4f}")

# 分析不同类型特征的贡献
print(f"\n特征类型分析:")
linear_terms = importance_df[~importance_df['特征'].str.contains(' ')]
quadratic_terms = importance_df[importance_df['特征'].str.contains('^2')]
interaction_terms = importance_df[importance_df['特征'].str.contains(' ') & 
                                 ~importance_df['特征'].str.contains('^2')]

print(f"线性项贡献: {linear_terms['重要性'].sum():.1f}")
print(f"二次项贡献: {quadratic_terms['重要性'].sum():.1f}")
print(f"交互项贡献: {interaction_terms['重要性'].sum():.1f}")

print("\n" + "=" * 80)
print("构建最终数学模型")
print("=" * 80)

# 选择最重要的项构建简化模型
n_top_features = 10
top_feature_names = top_features['特征'].tolist()[:n_top_features]
top_feature_indices = [list(feature_names).index(name) for name in top_feature_names]

# 重新训练简化模型
X_train_simplified = X_train_poly[:, top_feature_indices]
X_test_simplified = X_test_poly[:, top_feature_indices]

simplified_model = Ridge(alpha=0.1)
simplified_model.fit(X_train_simplified, y_train)

y_test_pred_simplified = simplified_model.predict(X_test_simplified)
simplified_r2 = r2_score(y_test, y_test_pred_simplified)
simplified_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_simplified))

print(f"简化模型性能 (使用{n_top_features}个最重要特征):")
print(f"R² = {simplified_r2:.4f}")
print(f"RMSE = {simplified_rmse:.1f} 米")

print(f"\n简化数学模型公式:")
print("visibility_mor_raw = β₀ + Σ(βᵢ × 特征ᵢ)")
print(f"\n其中:")
print(f"β₀ (截距) = {simplified_model.intercept_:.2f}")

# 构建公式
formula_parts = [f"{simplified_model.intercept_:.2f}"]
for i, (feature_name, coef) in enumerate(zip(top_feature_names, simplified_model.coef_)):
    sign = "+" if coef >= 0 else ""
    formula_parts.append(f"{sign}{coef:.2f}×{feature_name}")
    print(f"β{i+1} ({feature_name}): {coef:.6f}")

print(f"\n完整简化公式:")
print("visibility_mor_raw = " + " ".join(formula_parts))

print("\n" + "=" * 80)
print("物理意义解释")
print("=" * 80)

# 分析主要特征的物理意义
physical_interpretation = {
    'high_freq_ratio': '高频信息比例 - 反映图像细节清晰度',
    'high_freq_ratio^2': '高频比例二次项 - 非线性清晰度效应',
    'weather_temperature_c': '温度 - 影响雾的形成和消散',
    'weather_humidity_pct': '湿度 - 雾形成的关键气象要素',
    'laplacian_var': '拉普拉斯方差 - 图像锐度指标',
    'wind_wind_speed_10m': '风速 - 影响雾的输送和消散'
}

print("主要特征的物理意义:")
for feature in top_feature_names[:8]:
    base_feature = feature.split(' ')[0].split('^')[0]  # 获取基础特征名
    if base_feature in physical_interpretation:
        interpretation = physical_interpretation[base_feature]
        coef = top_features[top_features['特征'] == feature]['系数'].iloc[0]
        effect = "增加能见度" if coef > 0 else "降低能见度"
        print(f"• {feature}: {interpretation} → {effect}")

print("\n" + "=" * 80)
print("模型验证和误差分析")
print("=" * 80)

# 残差分析
residuals = y_test - y_test_pred
print(f"残差统计:")
print(f"均值: {residuals.mean():.2f} 米")
print(f"标准差: {residuals.std():.2f} 米")
print(f"偏度: {residuals.skew():.3f}")
print(f"峰度: {residuals.kurtosis():.3f}")

# 不同能见度范围的误差分析
low_vis_mask = y_test <= 1000
mid_vis_mask = (y_test > 1000) & (y_test <= 5000)
high_vis_mask = y_test > 5000

print(f"\n不同能见度范围的误差分析:")
for mask, name in [(low_vis_mask, "低能见度(<1000m)"), 
                   (mid_vis_mask, "中能见度(1000-5000m)"), 
                   (high_vis_mask, "高能见度(>5000m)")]:
    if mask.sum() > 0:
        range_mae = mean_absolute_error(y_test[mask], y_test_pred[mask])
        range_mape = np.mean(np.abs((y_test[mask] - y_test_pred[mask]) / y_test[mask])) * 100
        print(f"{name}: MAE={range_mae:.1f}米, MAPE={range_mape:.1f}%, 样本数={mask.sum()}")

# 综合可视化
plt.figure(figsize=(20, 12))

# 1. 预测vs真实值
plt.subplot(2, 4, 1)
plt.scatter(y_test, y_test_pred, alpha=0.6, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('真实能见度 (米)')
plt.ylabel('预测能见度 (米)')
plt.title(f'多项式回归模型\nR²={test_r2:.3f}, RMSE={test_rmse:.0f}m')

# 2. 残差分布
plt.subplot(2, 4, 2)
plt.hist(residuals, bins=50, alpha=0.7, color='green', edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--')
plt.xlabel('残差 (米)')
plt.ylabel('频次')
plt.title('残差分布')

# 3. 预测值vs残差
plt.subplot(2, 4, 3)
plt.scatter(y_test_pred, residuals, alpha=0.6, color='orange')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('预测值 (米)')
plt.ylabel('残差 (米)')
plt.title('预测值 vs 残差')

# 4. Q-Q图
from scipy import stats
plt.subplot(2, 4, 4)
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('残差正态性检验')

# 5. 特征重要性
plt.subplot(2, 4, 5)
top_10_features = top_features.head(10)
plt.barh(range(len(top_10_features)), top_10_features['重要性'], 
         color='purple', alpha=0.7)
plt.yticks(range(len(top_10_features)), 
           [f.replace('weather_', '').replace('_', ' ')[:15] + '...' if len(f) > 18 else f.replace('weather_', '').replace('_', ' ') 
            for f in top_10_features['特征']])
plt.xlabel('重要性 (|系数|)')
plt.title('特征重要性 Top 10')

# 6. 不同能见度范围的性能
plt.subplot(2, 4, 6)
ranges = ['低(<1000)', '中(1000-5000)', '高(>5000)']
range_maes = []
for mask in [low_vis_mask, mid_vis_mask, high_vis_mask]:
    if mask.sum() > 0:
        range_mae = mean_absolute_error(y_test[mask], y_test_pred[mask])
        range_maes.append(range_mae)
    else:
        range_maes.append(0)

plt.bar(ranges, range_maes, color=['red', 'orange', 'green'], alpha=0.7)
plt.ylabel('MAE (米)')
plt.title('不同能见度范围误差')
plt.xticks(rotation=45)

# 7. 模型改进对比
plt.subplot(2, 4, 7)
model_names = ['线性回归', '多项式回归', '简化多项式']
model_r2s = [0.8496, test_r2, simplified_r2]  # 从之前结果获得
plt.bar(model_names, model_r2s, color=['blue', 'red', 'green'], alpha=0.7)
plt.ylabel('R²')
plt.title('模型性能对比')
plt.xticks(rotation=45)
for i, r2 in enumerate(model_r2s):
    plt.text(i, r2 + 0.01, f'{r2:.3f}', ha='center', va='bottom')

# 8. 应用价值评估
plt.subplot(2, 4, 8)
plt.text(0.1, 0.9, '模型应用价值评估', fontsize=14, fontweight='bold', 
         transform=plt.gca().transAxes)
plt.text(0.1, 0.8, f'• 预测精度: R²={test_r2:.3f}', fontsize=10, 
         transform=plt.gca().transAxes)
plt.text(0.1, 0.7, f'• 平均误差: {test_mae:.0f}米', fontsize=10, 
         transform=plt.gca().transAxes)
plt.text(0.1, 0.6, f'• 相对误差: {mape:.1f}%', fontsize=10, 
         transform=plt.gca().transAxes)
plt.text(0.1, 0.5, '• 优势:', fontsize=12, fontweight='bold', 
         transform=plt.gca().transAxes)
plt.text(0.1, 0.4, '  - 非线性关系建模', fontsize=10, 
         transform=plt.gca().transAxes)
plt.text(0.1, 0.3, '  - 特征交互效应', fontsize=10, 
         transform=plt.gca().transAxes)
plt.text(0.1, 0.2, '  - 明确数学公式', fontsize=10, 
         transform=plt.gca().transAxes)
plt.text(0.1, 0.1, '  - 可解释性强', fontsize=10, 
         transform=plt.gca().transAxes)
plt.axis('off')

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("最终建模方案总结")
print("=" * 80)

print("【推荐方案】: 多项式回归模型")
print(f"• 模型类型: 二次多项式回归 + Ridge正则化")
print(f"• 核心特征: 5个 (图像特征2个 + 气象特征3个)")
print(f"• 扩展特征: 20个 (包含二次项和交互项)")
print(f"• 关键创新:")
print(f"  - 非线性关系: 考虑特征的二次效应")
print(f"  - 交互效应: 图像与气象要素的相互作用")
print(f"  - 正则化: 防止过拟合，提升泛化能力")

print(f"\n【模型性能】:")
print(f"• R² = {test_r2:.4f} (解释{test_r2*100:.1f}%的变异)")
print(f"• RMSE = {test_rmse:.1f}米 (平均误差)")
print(f"• MAE = {test_mae:.1f}米 (绝对误差)")
print(f"• MAPE = {mape:.1f}% (相对误差)")

print(f"\n【实际应用价值】:")
print(f"• 精度提升: 相比线性模型提升{(test_r2-0.8496)/0.8496*100:.1f}%")
print(f"• 成本效益: 仅需摄像头和基础气象数据")
print(f"• 实时性: 线性计算，响应速度快")
print(f"• 可靠性: 物理意义明确，结果可解释")

print(f"\n【建议应用场景】:")
print("• 机场跑道能见度实时监测")
print("• 高速公路雾况预警系统") 
print("• 港口航运气象服务")
print("• 气象台能见度预报辅助")

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.ensemble import GradientBoostingRegressor
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("创新建模方案：大气光学理论 + 高级数学方法")
print("=" * 80)

# 模拟数据（若无外部数据）
try:
    if 'df' not in globals():
        print("未检测到全局数据框df，生成模拟数据...")
        np.random.seed(42)
        n_samples = 1000
        df = pd.DataFrame({
            'laplacian_var': np.random.normal(0, 1, n_samples),
            'high_freq_ratio': np.random.normal(0, 1, n_samples),
            'weather_humidity_pct': np.random.uniform(20, 100, n_samples),
            'weather_temperature_c': np.random.uniform(-10, 35, n_samples),
            'wind_wind_speed_10m': np.random.uniform(0, 20, n_samples),
            'visibility_mor_raw': np.random.uniform(50, 15000, n_samples),
            'timestamp': pd.date_range('2023-01-01', periods=n_samples, freq='H')
        })
except Exception as e:
    print(f"数据初始化错误: {e}")
    raise

# 数据准备
required_columns = ['laplacian_var', 'high_freq_ratio', 'weather_humidity_pct', 
                    'weather_temperature_c', 'wind_wind_speed_10m', 'visibility_mor_raw', 'timestamp']
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    raise ValueError(f"数据缺少以下列: {missing_cols}")

print(f"数据形状: {df.shape}")
key_features = required_columns[:-2]  # 除去 'visibility_mor_raw' 和 'timestamp'

X = df[key_features].copy()
y = df['visibility_mor_raw'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("数据准备完成")

# 创新方案一：大气光学模型
print("\n【创新方案一：大气光学理论建模】")
def atmospheric_optics_model(params, X_img, X_weather):
    α1, α2, α3, α4, α5, β1, β2, γ1, γ2, γ3 = params
    img_contrib = α1 * X_img[:, 0] + α2 * X_img[:, 1]
    weather_contrib = α3 * X_weather[:, 0] + α4 * X_weather[:, 1] + α5 * X_weather[:, 2]
    nonlinear_terms = β1 * X_weather[:, 0] * X_weather[:, 1] + β2 * X_img[:, 0] * X_weather[:, 0]
    sigma_total = np.abs(img_contrib + weather_contrib + nonlinear_terms) + γ1
    sigma_total = np.maximum(sigma_total, 1e-6)
    mor = np.clip(3.912 / sigma_total, 10, 20000)
    return mor

def physics_objective(params, X_img, X_weather, y_true):
    y_pred = atmospheric_optics_model(params, X_img, X_weather)
    weights = 1.0 / (y_true + 100)
    return np.mean(weights * (y_pred - y_true) ** 2) + 0.001 * np.sum(params ** 2)

X_img_train = X_train_scaled[:, :2]
X_weather_train = X_train_scaled[:, 2:]
X_img_test = X_test_scaled[:, :2]
X_weather_test = X_test_scaled[:, 2:]

bounds = [(-2, 2)] * 10
result_de = differential_evolution(physics_objective, bounds, 
                                   args=(X_img_train, X_weather_train, y_train.values),
                                   seed=42, maxiter=50)
if result_de.success:
    optimal_params = result_de.x
    y_pred_physics = atmospheric_optics_model(optimal_params, X_img_test, X_weather_test)
    r2_physics = r2_score(y_test, y_pred_physics)
    rmse_physics = np.sqrt(mean_squared_error(y_test, y_pred_physics))
    print(f"大气光学模型 R² = {r2_physics:.4f}, RMSE = {rmse_physics:.1f} 米")
else:
    r2_physics = 0
    rmse_physics = 0
    y_pred_physics = np.zeros_like(y_test)

# 创新方案二：高斯过程回归
print("\n【创新方案二：高斯过程回归】")
sample_size = min(800, len(X_train_scaled))
sample_indices = np.random.choice(len(X_train_scaled), size=sample_size, replace=False)
X_sample = X_train_scaled[sample_indices]
y_sample = y_train.iloc[sample_indices].values

kernel = C(1.0, (1e-3, 1e3)) * RBF([1.0]*5, (1e-2, 1e2)) + WhiteKernel(1.0, (1e-10, 1e1))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=2, alpha=1e-6)
gp.fit(X_sample, y_sample)

test_sample_size = min(200, len(X_test_scaled))
test_indices = np.random.choice(len(X_test_scaled), size=test_sample_size, replace=False)
X_test_gp = X_test_scaled[test_indices]
y_test_gp = y_test.iloc[test_indices].values
y_pred_gp, y_std_gp = gp.predict(X_test_gp, return_std=True)
r2_gp = r2_score(y_test_gp, y_pred_gp)
rmse_gp = np.sqrt(mean_squared_error(y_test_gp, y_pred_gp))
print(f"高斯过程回归 R² = {r2_gp:.4f}, RMSE = {rmse_gp:.1f} 米")

# 创新方案三：时间序列动态模型
print("\n【创新方案三：时间序列动态模型】")
df_sorted = df.sort_values('timestamp').reset_index(drop=True)

def create_time_features(data, lag_steps=[1, 2, 3]):
    data_time = data.copy()
    for lag in lag_steps:
        for col in ['visibility_mor_raw', 'weather_humidity_pct', 'laplacian_var']:
            data_time[f'{col}_lag{lag}'] = data[col].shift(lag)
    for window in [3, 5]:
        for col in ['weather_humidity_pct', 'weather_temperature_c']:
            data_time[f'{col}_ma{window}'] = data[col].rolling(window=window).mean()
    data_time = data_time.dropna()
    return data_time

df_time = create_time_features(df_sorted)
time_features = [col for col in df_time.columns if col in key_features or 'lag' in col or 'ma' in col]
X_time = df_time[time_features]
y_time = df_time['visibility_mor_raw']
split_idx = int(0.8 * len(X_time))
X_train_time = X_time.iloc[:split_idx]
X_test_time = X_time.iloc[split_idx:]
y_train_time = y_time.iloc[:split_idx]
y_test_time = y_time.iloc[split_idx:]

scaler_time = StandardScaler()
X_train_time_scaled = scaler_time.fit_transform(X_train_time)
X_test_time_scaled = scaler_time.transform(X_test_time)

time_model = GradientBoostingRegressor(n_estimators=100, max_depth=6, random_state=42)
time_model.fit(X_train_time_scaled, y_train_time)
y_pred_time = time_model.predict(X_test_time_scaled)
r2_time = r2_score(y_test_time, y_pred_time)
rmse_time = np.sqrt(mean_squared_error(y_test_time, y_pred_time))
print(f"时间序列模型 R² = {r2_time:.4f}, RMSE = {rmse_time:.1f} 米")

# 创新方案四：物理约束优化模型
print("\n【创新方案四：物理约束优化模型】")
def physics_constrained_loss(params, X, y):
    y_pred = X @ params[:-1] + params[-1]
    humidity_idx = 2
    high_humidity_mask = X[:, humidity_idx] > 1.5
    high_humidity_penalty = np.sum(np.maximum(0, y_pred[high_humidity_mask] - 2000) ** 2) * 0.001 if np.any(high_humidity_mask) else 0
    range_penalty = (np.sum(np.maximum(0, y_pred - 15000) ** 2) + np.sum(np.maximum(0, 10 - y_pred) ** 2)) * 0.001
    mse_loss = np.mean((y_pred - y) ** 2)
    return mse_loss + high_humidity_penalty + range_penalty

n_features = X_train_scaled.shape[1]
bounds_constrained = [(-10, 10)] * (n_features + 1)
result_constrained = differential_evolution(physics_constrained_loss, bounds_constrained, 
                                            args=(X_train_scaled, y_train.values), seed=42, maxiter=50)
if result_constrained.success:
    constrained_params = result_constrained.x
    y_pred_constrained = X_test_scaled @ constrained_params[:-1] + constrained_params[-1]
    r2_constrained = r2_score(y_test, y_pred_constrained)
    rmse_constrained = np.sqrt(mean_squared_error(y_test, y_pred_constrained))
    print(f"物理约束模型 R² = {r2_constrained:.4f}, RMSE = {rmse_constrained:.1f} 米")
else:
    r2_constrained = 0
    y_pred_constrained = np.zeros_like(y_test)

# 创新方案五：物理约束优化模型（扩展）
print("\n【创新方案五：物理约束优化模型】")
def physics_constrained_model(params, X):
    a1, a2, a3, a4, a5, b1, b2, b3 = params
    visibility = (a1 * X[:, 0] + a2 * X[:, 1] + a3 * X[:, 2] + a4 * X[:, 3] + a5 * X[:, 4] + 
                  b1 * X[:, 2] * X[:, 3] + b2 * X[:, 0] * X[:, 1] + b3)
    humidity_penalty = np.where(X[:, 2] > 1.5, visibility * 0.5, visibility)
    return np.maximum(humidity_penalty, 50)

def objective_five(params, X, y):
    y_pred = physics_constrained_model(params, X)
    return np.mean((y_pred - y) ** 2)

bounds_five = [(-5, 5)] * 8
result_five = differential_evolution(objective_five, bounds_five, 
                                     args=(X_train_scaled, y_train.values), seed=42, maxiter=50)
if result_five.success:
    params_five = result_five.x
    y_pred_five = physics_constrained_model(params_five, X_test_scaled)
    r2_five = r2_score(y_test, y_pred_five)
    rmse_five = np.sqrt(mean_squared_error(y_test, y_pred_five))
    print(f"扩展物理约束模型 R² = {r2_five:.4f}, RMSE = {rmse_five:.1f} 米")
else:
    r2_five = 0
    y_pred_five = np.zeros_like(y_test)

# 可视化
plt.figure(figsize=(20, 15))
model_names = ['物理光学', '高斯过程', '时间序列', '物理约束', '扩展约束']
r2_values = [r2_physics, r2_gp, r2_time, r2_constrained, r2_five]
colors = ['red', 'blue', 'green', 'orange', 'purple']

plt.subplot(2, 3, 1)
bars = plt.bar(model_names, r2_values, color=colors)
plt.ylabel('R²')
plt.title('模型性能对比')
plt.xticks(rotation=45)
for bar, r2 in zip(bars, r2_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{r2:.3f}', ha='center')

plt.subplot(2, 3, 2)
plt.scatter(y_test, y_pred_physics, alpha=0.6, color='red')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('真实能见度')
plt.ylabel('预测能见度')
plt.title(f'物理光学模型 R²={r2_physics:.3f}')

plt.subplot(2, 3, 3)
plt.errorbar(y_test_gp[:50], y_pred_gp[:50], yerr=y_std_gp[:50], fmt='o', alpha=0.6, color='blue')
plt.plot([y_test_gp.min(), y_test_gp.max()], [y_test_gp.min(), y_test_gp.max()], 'k--')
plt.xlabel('真实能见度')
plt.ylabel('预测能见度±不确定性')
plt.title(f'高斯过程回归 R²={r2_gp:.3f}')

plt.subplot(2, 3, 4)
time_range = range(min(100, len(y_test_time)))
plt.plot(time_range, y_test_time.iloc[:len(time_range)], 'b-', label='真实值')
plt.plot(time_range, y_pred_time[:len(time_range)], 'r-', label='预测值')
plt.xlabel('时间步')
plt.ylabel('能见度')
plt.title('时间序列预测')
plt.legend()

plt.subplot(2, 3, 5)
plt.scatter(y_test, y_pred_five, alpha=0.6, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('真实能见度')
plt.ylabel('预测能见度')
plt.title(f'扩展物理约束模型 R²={r2_five:.3f}')

plt.tight_layout()
plt.show()

# 总结
print("=" * 80)
print("创新建模方案总结")
print("=" * 80)
print("1. 大气光学理论模型: 基于Beer-Lambert定律的物理建模")
print("2. 高斯过程回归: 提供预测不确定性量化")
print("3. 时间序列动态模型: 考虑雾的时间连续性")
print("4. 物理约束模型: 加入领域知识约束")
print("5. 扩展物理约束模型: 增强物理规则的优化")
print(f"\n最佳模型性能: R² = {max(r2_values):.4f}")
print("这些方法显著提升了模型的科学性和实用性！")